# Importation of necessary Python Libs

## Maths or data structure-related libraries

In [5]:
import math
import numpy as np
import random
import pandas as pd
from collections import deque, namedtuple
from typing import Tuple, List, Set, Dict

## Libraries for visualization/record tracking

In [2]:
import matplotlib.pyplot as plt
import tqdm

## Libraries for machine leanring

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler

# Implementation of the VAEQL algoritm

The link to my manuscript: <a>https://www.overleaf.com/project/67cb575babefcc1067d01469</a>

## Defining the miscelaneous methods

### Defining the training method depending on the hardware device

In [3]:
def training_func():
    device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu" # CUDA doesn't work with the AMD GPUs of MacBook M1
    if device == "cuda:0":
        print("Training on the GPU")
    else:
        print("Training on the CPU")

    ENV = namedtuple('env', ('name', 'n_actions', 'encoding_dim'))

### Copy and paste the benchmark dataframe pre-processing methods from the previous experiments

In [ ]:
def identify_binary_and_numerical_features(df: pd.DataFrame) -> Tuple[List[str]]:
     # 1. Identify “binary” columns (unique values ⊆ {0,1})
    categorical_cols = [col for col in df.columns if match(r".+-is_.+", col)]
    
    # 2. All the other numeric columns
    numeric_cols = list(set(df.columns) - set(categorical_cols))

    # return to a tuple of two lists of column names
    return numeric_cols, categorical_cols

In [ ]:
def normalize_numerical_features(df: pd.DataFrame, num_features: List[str]) -> pd.DataFrame:
    # Create a MinMaxScaler instance
    scaler = MinMaxScaler()

    # Fit and transform the numerical features to the range [0, 1]
    df[num_features] = scaler.fit_transform(df[num_features])
    
    return df

In [ ]:
def generate_masks_for_missingness(
    original_df: pd.DataFrame,  
    amputed_df: pd.DataFrame,
    num_feats: List[str],
    cat_feats: List[str],
    imputed_original_df: pd.DataFrame = None,
) -> Tuple[np.ndarray, np.ndarray]:

    if not type(imputed_original_df) == pd.DataFrame:
        print("No missingness!")
        imputed_original_df = original_df.copy()

    # Check if the dataframes match in shape
    if not original_df.shape == imputed_original_df.shape == amputed_df.shape:
        raise Exception("Sorry, the three dataframes do not match in shape")

    # Check if the dataframes have identical column names
    if not set(original_df.columns) == set(imputed_original_df.columns) == set(amputed_df.columns):
        raise Exception("Sorry, the three dataframes do not match in column names")

    # Initialize the maps for numerical and categorical features
    num_feat_map = np.zeros(original_df[num_feats].shape, dtype=int)
    cat_feat_map = np.zeros(original_df[cat_feats].shape, dtype=int)
    
    # Generate the missingness map for numerical features
    for i, feat in enumerate(num_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]
        
        # 1: missing in the original dataset, regardless of whether it is amputed
        num_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        num_feat_map[:, i] = np.where(amputed_col.isna(), 2, num_feat_map[:, i])  # 2 if amputed
    
    # Generate the missingness map for one-hot-encoded categorical features
    for i, feat in enumerate(cat_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]
        
        # 1: missing in the original dataset, regardless of whether it is amputed
        cat_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        cat_feat_map[:, i] = np.where(amputed_col.isna(), 2, cat_feat_map[:, i])  # 2 if amputed

    # Return the maps as a tuple of two arrays
    return num_feat_map, cat_feat_map

## Defining the Variational Autoencoder part of the VAEQL algorithm

### The trainer class

In [4]:
class MaskedVAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=(128, 64)):
        super().__init__()
        # Encoder: input concatenated with binary observed-mask
        enc_modules = []
        prev_dim = input_dim + input_dim  # raw X + one-hot mask indicator (observed)
        for h in hidden_dims:
            enc_modules += [nn.Linear(prev_dim, h), nn.ReLU(inplace=True)]
            prev_dim = h
        self.encoder = nn.Sequential(*enc_modules)
        self.fc_mu = nn.Linear(prev_dim, latent_dim)
        self.fc_logvar = nn.Linear(prev_dim, latent_dim)

        # Decoder: latent z back to original feature dimension
        dec_modules = []
        prev_dim = latent_dim
        for h in reversed(hidden_dims):
            dec_modules += [nn.Linear(prev_dim, h), nn.ReLU(inplace=True)]
            prev_dim = h
        dec_modules += [nn.Linear(prev_dim, input_dim)]
        self.decoder = nn.Sequential(*dec_modules)

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        """
        x:      (batch, D) amputed+normalized inputs
        m:      (batch, D) mask with values {0,1,2}
                0=originally present, 1=originally missing, 2=artificially amputed
        """
        # binary observed-mask: 1 for entries originally present (m==0)
        m_obs = (m == 0).float()
        # encode
        h_enc = torch.cat([x, m_obs], dim=1)
        h = self.encoder(h_enc)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        # reparameterize
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        # decode
        x_hat = torch.sigmoid(self.decoder(z))
        return x_hat, mu, logvar


### The loss function

In [ ]:
def masked_vae_loss(x_hat: torch.Tensor,
                    x: torch.Tensor,
                    mu: torch.Tensor,
                    logvar: torch.Tensor,
                    m: torch.Tensor,
                    beta: float = 1.0,
                    eps: float = 1e-8) -> torch.Tensor:
    """
    Reconstruction loss only computed on entries with m==0 (originally observed).
    KL term as usual.
    """
    # elementwise BCE
    rec_el = F.binary_cross_entropy(x_hat, x, reduction='none')
    # mask for original entries
    m_obs = (m == 0).float()
    # apply mask
    rec_masked = rec_el * m_obs
    # normalized by number of observed entries
    rec = rec_masked.sum() / (m_obs.sum() + eps)

    # KL divergence
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()

    return rec + beta * kl

### Testing on the training part

## Defining the MDP Q-learning part of the VAEQL algorithm

In [33]:
class MDP_Q_learner():
    def init(self, 
            device:str, 
            input_df: DataFrame,
            states: namedtuple, 
            actions: namedtuple,
            gamma: float=0.9, # discount factor
            alpha = 1e-2, # learning rate
            epsilon=0.2,
            ):
        # externally defined input variables, staying constant
        self.DF = input_df
        self.STATES = states
        self.ACTIONS = actions
        self.DEVICE = device
        # variables that stay constantly updated during Q-learning
        self.state = self.calculate_state() # using self.DF
        self.state_next = None
        # on-policy learning, if it doesn't work, might switch to off-policy learning with replay buffer later
        self.policy_q = torch.zeros(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE) 
        # alternative choice
        #self.policy_q = torch.rand(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE) 
        # provisional Target-Q table for off-policy learning
        #self.target_q = torch.rand(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        #self.target_q = torch.zeros(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE) 
    
    def calculate_state(self) -> torch.Tensor:
        pass
    
    def update_state(self) -> None:
        pass

    def calculate_reward(self) -> float:
        pass

    def update_policy_Q_table(self) -> None:
        pass

    def start_new_episode(self, state) -> None:
        self.state = self.calculate_state()
        self.state_next = None
        # in the case of off-policy learning
        #self.target_q = self.policy_q.detach().clone()

    def epsilon_greedy_action(self, state) -> torch.Tensor:
        pass

### Defining the overall data loader, pre-processor and trainer class

# Testing the trainer class above on the five pre-processed datasets